In [ ]:
# 检查本 notebook 所需第三方库是否已安装，并打印其版本号，便于排查环境/依赖问题
from importlib.metadata import version

# 需要检查版本的关键依赖包列表
pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# 开关：True 表示加载指令微调版模型 gemma-3-270m-it（适合直接问答/对话场景）
# False 表示加载基础预训练版模型 gemma-3-270m（未做指令微调，偏重文本续写）
USE_INSTRUCT_MODEL = True

In [ ]:
# 本单元定义 Gemma3 的完整网络结构（FeedForward、RMSNorm、RoPE、分组查询注意力+QK-Norm、
# TransformerBlock、Gemma3Model）。Gemma3 相对标准 Transformer 的关键特点：
# 1) 用 RMSNorm 替代 LayerNorm，且实现上是 weight 从 0 初始化、前向用 (1+weight) 缩放；
# 2) 用 RoPE（旋转位置编码）而非可学习/绝对位置编码；
# 3) 用分组查询注意力（GQA，这里 n_kv_groups=1 即 Multi-Query Attention）节省 KV 缓存；
# 4) 引入 QK-Norm：对每个注意力头的 Q/K 分别做 RMSNorm，用来替代 Gemma2 中的
#    logit 软上限（softcap）——本实现中注意力分数不再做 tanh 软截断；
# 5) 局部（滑动窗口）注意力层与全局注意力层交替出现，分别使用不同的 RoPE 频率基数；
# 6) 词嵌入之后要乘以 sqrt(emb_dim) 进行缩放。
import torch
import torch.nn as nn


# FeedForward：SwiGLU 风格的门控前馈网络（GLU 变体），而非普通的 Linear+激活+Linear，
# 所有线性层均不带 bias，这与 Llama/Gemma 等现代 LLM 的做法一致
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # fc1：门控（gate）分支，emb_dim -> hidden_dim
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        # fc2：上投影（up）分支，emb_dim -> hidden_dim，与 fc1 逐元素相乘构成门控
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        # fc3：下投影（down）分支，hidden_dim -> emb_dim，把门控后的结果映射回原始维度
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        # x_fc1、x_fc2 形状均为 (b, seq_len, hidden_dim)
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        # 门控机制：对 fc1 输出做 tanh 近似的 GELU 激活，再与 fc2 的线性输出逐元素相乘
        # （即 GLU：Gated Linear Unit），比单纯的 激活(Linear(x)) 表达能力更强
        x = nn.functional.gelu(x_fc1, approximate="tanh") * x_fc2
        return self.fc3(x)
# RMSNorm（均方根归一化）：只用均方根做缩放，不做去均值，比 LayerNorm 计算更省、无偏置，
# 是 Llama/Gemma/Mistral 等现代 LLM 常用的归一化方式
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False):
        super().__init__()
        self.eps = eps
        # Gemma3 stores zero-centered weights and uses (1 + weight) during forward
        # 权重从全 0 初始化，前向用 (1 + weight) 而非 weight 本身，
        # 这样训练初期 (1+0)=1 相当于恒等缩放，有利于训练稳定
        self.scale = nn.Parameter(torch.zeros(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        # Match HF Gemma3: compute norm in float32, then scale by (1 + w)
        # 记录输入的原始 dtype，方便最后转换回去
        input_dtype = x.dtype
        x_f = x.float()
        # 沿最后一维（通道/head_dim 维）计算均方值，即 RMS 中的“均方”部分
        var = x_f.pow(2).mean(dim=-1, keepdim=True)
        # 用 1/sqrt(均方 + eps) 缩放，实现均方根归一化（不减均值，区别于 LayerNorm）
        x_norm = x_f * torch.rsqrt(var + self.eps)
        # 关键点：用 (1 + scale) 而不是 scale 直接相乘，配合上面 scale 从 0 初始化的设计
        out = x_norm * (1.0 + self.scale.float())

        if self.shift is not None:
            out = out + self.shift.float()

        # 转换回输入原始精度（如 bfloat16），保持与模型其余部分 dtype 一致
        return out.to(input_dtype)
# 预计算 RoPE（旋转位置编码）的频率表。Gemma3 对局部（滑动窗口）注意力层和全局注意力层
# 使用不同的 theta_base（分别对应 rope_local_base 与 rope_base），全局层的 base 大得多，
# 从而能在很长的上下文长度下仍保持较好的位置区分度
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    # 逆频率 inv_freq = 1 / theta_base^(2i/head_dim)，i ∈ [0, head_dim/2)；
    # theta_base 越大，频率越低、旋转越“慢”，适合建模长距离依赖（用于全局注意力层）
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))

    # Generate position indices
    # 生成 [0, 1, ..., context_length-1] 的位置索引
    positions = torch.arange(context_length, dtype=dtype)

    # Compute the angles
    # 位置 × 频率 得到每个位置在每个频率分量上的旋转角度
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

    # Expand angles to match the head_dim
    # 把角度在最后一维复制一份拼接，扩展到完整 head_dim（RoPE 把 head_dim 一分为二使用同一组角度）
    angles = torch.cat([angles, angles], dim=1)  # Shape: (context_length, head_dim)

    # Precompute sine and cosine
    # 预先算好 cos/sin 并缓存，避免每次前向传播都重复计算三角函数
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


# 把 RoPE 旋转变换实际应用到 Q 或 K 张量上：将 head_dim 拆成前后两半 (x1, x2)，
# 通过 (x*cos + rotate_half(x)*sin) 的组合实现二维子空间上的“旋转”，从而把相对位置信息注入注意力
def apply_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    # 前一半维度
    x1 = x[..., : head_dim // 2]  # First half
    # 后一半维度
    x2 = x[..., head_dim // 2 :]  # Second half

    # Adjust sin and cos shapes
    # 按当前序列长度截取预计算的 cos/sin，并广播出 batch、head 两个维度
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    # 构造“旋转后”向量：把后半部分取负号放到前面、前半部分放到后面
    rotated = torch.cat((-x2, x1), dim=-1)
    # 标准 RoPE 公式：x_rotated = x*cos + rotate_half(x)*sin
    x_rotated = (x * cos) + (rotated * sin)

    # It's ok to use lower-precision after applying cos and sin rotation
    return x_rotated.to(dtype=x.dtype)
# 分组查询注意力（Grouped-Query Attention, GQA）：Query 用 num_heads 个头，
# 但 Key/Value 只用较少的 num_kv_groups 组（本配置为 1，即退化为 Multi-Query Attention），
# 可显著降低 KV 缓存的显存/带宽占用。此外还支持可选的 QK-Norm（对每个头的 Q、K 分别做 RMSNorm），
# 这是 Gemma3 用来替代 Gemma2 中“注意力 logit 软上限（softcap）”的数值稳定手段——
# 通过在计算注意力分数前归一化 Q/K 的尺度，避免出现极端注意力分数，因此这里不再需要 tanh 软截断
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False,
        query_pre_attn_scalar=None, dtype=None,
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        # group_size：每个 KV 组要被多少个 Query 头共享（重复使用）
        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        # Query 投影：d_in -> num_heads * head_dim（每个头独立的 Q）
        self.W_query = nn.Linear(d_in, self.d_out, bias=False, dtype=dtype)
        # Key 投影：d_in -> num_kv_groups * head_dim（远小于 Q，节省参数与显存）
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        # Value 投影，与 Key 同理，使用相同数量的 KV 组
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)

        # 输出投影：把多头拼接后的结果映射回 d_in
        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        # QK-Norm：Gemma3 的关键特性之一——对每个注意力头的 Q/K（维度为 head_dim）分别做 RMSNorm，
        # 在应用 RoPE 之前执行，用于稳定注意力 logits 的数值范围
        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

        # 缩放系数：Gemma3 用配置中显式给定的 query_pre_attn_scalar 计算 scaling，而不是直接用 head_dim，
        # 便于不同规模模型复用同一套缩放策略（此配置下 query_pre_attn_scalar=256=head_dim，
        # 数值上等价于标准的 1/sqrt(head_dim) 缩放）
        if query_pre_attn_scalar is not None:
            self.scaling = (query_pre_attn_scalar) ** -0.5
        else:
            self.scaling = (head_dim) ** -0.5


    # x: (b, num_tokens, d_in)；mask/cos/sin 由调用方传入，
    # 分别对应当前层要用的“全局”或“局部”一整套掩码与 RoPE 频率表
    def forward(self, x, mask, cos, sin):
        b, num_tokens, _ = x.shape

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # Reshape
        # reshape 并转置：把 num_heads 换到第 2 维，得到 (b, num_heads, num_tokens, head_dim) 的多头格式
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # Optional normalization
        # 若启用 QK-Norm，则在做 RoPE 之前先对每个头的 Q/K 做 RMSNorm（在 head_dim 维度上归一化）
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys = self.k_norm(keys)

        # Apply RoPE
        queries = apply_rope(queries, cos, sin)
        keys = apply_rope(keys, cos, sin)

        # Expand K and V to match number of heads
        # K/V 只有 num_kv_groups 组，这里用 repeat_interleave 把每组沿头维度复制 group_size 次，
        # 使 K/V 的头数与 Q 对齐，才能逐头计算注意力
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # Scale queries
        # 对 Query 整体乘以缩放系数（与在点积后再缩放数值等价，但精度上更稳定）
        queries = queries * self.scaling

        # Attention
        # 注意力得分，形状 (b, num_heads, num_tokens, num_tokens)。
        # 注意：Gemma3 这里没有像 Gemma2 那样对 attn_scores 做 tanh 软上限（logit softcap），
        # 而是依靠前面的 QK-Norm 与缩放来控制数值范围
        attn_scores = queries @ keys.transpose(2, 3)
        # 用 mask（局部/全局）把不允许关注的位置（未来 token，或滑窗之外的过远历史）填充为 -inf，
        # softmax 后这些位置权重为 0，从而实现因果/局部约束
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(attn_scores, dim=-1)

        # 加权求和得到上下文向量，再转置、reshape 回 (b, num_tokens, num_heads*head_dim)，
        # 最后经输出投影融合各头信息
        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context)
# Transformer 块：采用“双重归一化（sandwich norm）”结构——注意力和前馈子层各自在
# 输入端、输出端都有一次 RMSNorm（input_layernorm/post_attention_layernorm 对应注意力子层，
# pre_feedforward_layernorm/post_feedforward_layernorm 对应前馈子层），
# 比标准 Pre-LN 多了输出端归一化，有助于训练更深网络时保持数值稳定
class TransformerBlock(nn.Module):

    def __init__(self, cfg, attn_type):
        super().__init__()
        self.attn_type = attn_type

        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_groups"],
            head_dim=cfg["head_dim"],
            qk_norm=cfg["qk_norm"],
            query_pre_attn_scalar=cfg["query_pre_attn_scalar"],
            dtype=cfg["dtype"],
        )
        self.ff = FeedForward(cfg)
        # 四个 RMSNorm 依次对应：注意力前、注意力后、前馈前、前馈后（即“sandwich”结构）
        self.input_layernorm = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.post_attention_layernorm = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.pre_feedforward_layernorm = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.post_feedforward_layernorm = RMSNorm(cfg["emb_dim"], eps=1e-6)

    # 前向同时接收全局/局部两套 mask 与 RoPE 表，具体用哪一套由本层的 attn_type 决定
    def forward(
        self,
        x,
        mask_global,
        mask_local,
        cos_global,
        sin_global,
        cos_local,
        sin_local,
    ):
        # Shortcut connection for attention block
        shortcut = x
        x = self.input_layernorm(x)

        # 根据当前层类型选择局部滑窗注意力还是全局注意力对应的 mask 与 RoPE 参数——
        # 这正是 Gemma3“局部/全局交替注意力”的核心开关
        if self.attn_type == "sliding_attention":
            attn_mask = mask_local
            cos = cos_local
            sin = sin_local
        else:
            attn_mask = mask_global
            cos = cos_global
            sin = sin_global

        x_attn = self.att(x, attn_mask, cos, sin)
        # 注意力输出先做一次归一化（post-norm），再与残差相加
        x_attn = self.post_attention_layernorm(x_attn)
        x = shortcut + x_attn

        # Shortcut connection for feed forward block
        shortcut = x
        # 前馈网络之前先做一次归一化（pre-norm）
        x_ffn = self.pre_feedforward_layernorm(x)
        x_ffn = self.ff(x_ffn)
        # 前馈网络输出后再做一次归一化（post-norm），然后才与残差相加
        x_ffn = self.post_feedforward_layernorm(x_ffn)
        x = shortcut + x_ffn
        return x
# 完整的 Gemma3 模型：词嵌入 -> N 个交替局部/全局注意力的 TransformerBlock -> 最终 RMSNorm ->
# 输出线性层（得到词表大小的 logits）。同时预先计算并缓存两套 RoPE（局部/全局）的 cos/sin
class Gemma3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # layer_types 逐层指定注意力类型（sliding_attention 或 full_attention），
        # 长度必须与层数一致
        assert cfg["layer_types"] is not None and len(cfg["layer_types"]) == cfg["n_layers"]

        # Main model parameters
        # 词嵌入表，形状 (vocab_size, emb_dim)
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        # 按 cfg["layer_types"] 中的顺序为每一层创建对应 attn_type 的 TransformerBlock，
        # 实现“大部分层用局部滑窗注意力、少数层用全局注意力”的交替结构
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg, attn_type)for attn_type in cfg["layer_types"]
        ])

        # 输出前的最终归一化层
        self.final_norm = RMSNorm(cfg["emb_dim"], eps=1e-6)
        # 输出投影：emb_dim -> vocab_size，得到每个位置对每个词的打分（logits）。
        # 若权重文件中没有单独的 lm_head 权重，后续 load_weights 会让它与词嵌入权重共享（weight tying）
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])
        self.cfg = cfg

        # Reusable utilities
        # 局部注意力层用较小的 theta_base，旋转频率更“密集”，更适合建模短距离依赖
        cos_local, sin_local = compute_rope_params(
            head_dim=cfg["head_dim"],
            theta_base=cfg["rope_local_base"],
            context_length=cfg["context_length"],
            dtype=torch.float32,
        )
        # 全局注意力层用远大得多的 theta_base，旋转频率更“稀疏”，
        # 能在很长上下文（如 32768）下仍保持较好的位置区分度
        cos_global, sin_global = compute_rope_params(
            head_dim=cfg["head_dim"],
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"],
            dtype=torch.float32,
        )
        # 用 register_buffer 保存这些预计算张量，使其可随 .to(device/dtype) 一起搬运，
        # 但不作为可训练参数；persistent=False 表示不写入 state_dict（可随时重新计算）
        self.register_buffer("cos_local", cos_local, persistent=False)
        self.register_buffer("sin_local", sin_local, persistent=False)
        self.register_buffer("cos_global", cos_global, persistent=False)
        self.register_buffer("sin_global", sin_global, persistent=False)

    # 构建两种注意力掩码：mask_global 是标准因果掩码（只屏蔽“未来”）；
    # mask_local 在此基础上再屏蔽“过远的历史”（超出 sliding_window），实现局部滑窗注意力
    def _create_masks(self, seq_len, device):
        # 先构造一个全 True 的方阵作为基础
        ones = torch.ones((seq_len, seq_len), dtype=torch.bool, device=device)

        # mask_global (future is masked: j > i)
        #     j:  0 1 2 3 4 5 6 7
        #  i
        #     0:  0 1 1 1 1 1 1 1
        #     1:  0 0 1 1 1 1 1 1
        #     2:  0 0 0 1 1 1 1 1
        #     3:  0 0 0 0 1 1 1 1
        #     4:  0 0 0 0 0 1 1 1
        #     5:  0 0 0 0 0 0 1 1
        #     6:  0 0 0 0 0 0 0 1
        #     7:  0 0 0 0 0 0 0 0
        # 上三角（不含对角线）为 True 表示 j>i 即“未来”位置，屏蔽掉即得标准因果掩码
        mask_global = torch.triu(ones, diagonal=1)

        # far_past (too far back is masked: i - j >= sliding_window)
        # where sliding_window = 4
        #     j:  0 1 2 3 4 5 6 7
        #  i
        #     0:  0 0 0 0 0 0 0 0
        #     1:  0 0 0 0 0 0 0 0
        #     2:  0 0 0 0 0 0 0 0
        #     3:  0 0 0 0 0 0 0 0
        #     4:  1 0 0 0 0 0 0 0
        #     5:  1 1 0 0 0 0 0 0
        #     6:  1 1 1 0 0 0 0 0
        #     7:  1 1 1 1 0 0 0 0
        # 利用转置后的上三角掩码，构造“i 与 j 的距离达到或超过 sliding_window”的过远历史掩码
        far_past = torch.triu(ones, diagonal=self.cfg["sliding_window"]).T

        # Local (sliding_window) = future OR far-past
        # mask_local
        #     j:  0 1 2 3 4 5 6 7
        # i
        # 0:      0 1 1 1 1 1 1 1
        # 1:      0 0 1 1 1 1 1 1
        # 2:      0 0 0 1 1 1 1 1
        # 3:      0 0 0 0 1 1 1 1
        # 4:      1 0 0 0 0 1 1 1
        # 5:      1 1 0 0 0 0 1 1
        # 6:      1 1 1 0 0 0 0 1
        # 7:      1 1 1 1 0 0 0 0
        # 局部掩码 = 因果掩码 ∪ 过远历史掩码：只能看当前及最近 sliding_window 个位置内的历史，且不能看未来
        mask_local = mask_global | far_past
        return mask_global, mask_local

    # 整个模型的前向传播
    def forward(self, input_ids):
        # Forward pass
        b, seq_len = input_ids.shape
        # 关键细节：Gemma 系列在词嵌入之后要乘以 sqrt(emb_dim) 进行缩放，
        # 目的是让嵌入的方差与网络其余部分的尺度匹配，从而稳定数值（标准 Transformer 通常没有这一步）
        x = self.tok_emb(input_ids) * (self.cfg["emb_dim"] ** 0.5)
        # 根据当前序列长度动态生成两种掩码
        mask_global, mask_local = self._create_masks(seq_len, x.device)

        # 依次通过每个 Transformer 块，把全局/局部两套 mask 与 RoPE 参数都传进去，
        # 由每层根据自身 attn_type 选择使用哪一套
        for block in self.blocks:
            x = block(
                x,
                mask_global=mask_global,
                mask_local=mask_local,
                cos_global=self.cos_global,
                sin_global=self.sin_global,
                cos_local=self.cos_local,
                sin_local=self.sin_local,
            )

        # 最后做一次 RMSNorm
        x = self.final_norm(x)
        # 计算 logits 前先把隐藏状态转换为 cfg 指定的 dtype（如 bfloat16），与输出层权重保持一致，
        # 投影到词表维度，得到形状 (b, seq_len, vocab_size) 的 logits
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

2. Initialize model

In [ ]:
# Gemma3-270M 模型的超参数配置，对应 HuggingFace 上 google/gemma-3-270m(-it) 的结构
GEMMA3_CONFIG_270M = {
    # 词表大小（Gemma3 使用较大的多语言词表）
    "vocab_size": 262_144,
    # 支持的最大上下文长度
    "context_length": 32_768,
    # 词嵌入/隐藏层维度
    "emb_dim": 640,
    # 注意力 Query 头数
    "n_heads": 4,
    # Transformer 层数
    "n_layers": 18,
    # 前馈网络（FeedForward）中间层维度
    "hidden_dim": 2048,
    # 每个注意力头的维度：注意这是独立配置的（640/4=160 ≠ 256），
    # 说明 Q/K/V 投影维度与 emb_dim、n_heads 解耦
    "head_dim": 256,
    # 是否对每个注意力头的 Q/K 做 RMSNorm（QK-Norm），Gemma3 的关键稳定性设计
    "qk_norm": True,
    # KV 分组数：为 1 表示所有 Query 头共享同一组 Key/Value（即 Multi-Query Attention，
    # 是 GQA 的极端情形），可大幅降低 KV 缓存显存占用
    "n_kv_groups": 1,
    # 局部（滑动窗口）注意力层 RoPE 的 theta 基数，数值较小，适合短距离位置建模
    "rope_local_base": 10_000.0,
    # 全局注意力层 RoPE 的 theta 基数，远大于 rope_local_base，用于支撑长上下文下的位置编码
    "rope_base": 1_000_000.0,
    # 局部滑窗注意力的窗口大小：每个 token 只能看到最近 512 个历史 token
    "sliding_window": 512,
      # 逐层指定注意力类型：大多数层为 sliding_attention（局部滑窗，开销小），
      # 每 6 层中有 1 层是 full_attention（全局注意力，可看到完整历史），
      # 形成局部/全局交替模式，兼顾效率与长距离建模能力，是 Gemma3 的标志性设计之一
      "layer_types": [
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention"
    ],
    # 模型参数的数据类型，用 bfloat16 节省显存/加速计算，同时保持较好的数值范围
    "dtype": torch.bfloat16,
    # 注意力缩放所用标量，GroupedQueryAttention 用它计算 scaling=scalar**-0.5
    # （此处等于 head_dim，效果上等价于标准 1/sqrt(head_dim) 缩放）
    "query_pre_attn_scalar": 256,
}
# 固定随机种子，保证模型参数初始化（加载预训练权重前）可复现
torch.manual_seed(123)
# 实例化模型（此时是随机初始化权重，稍后会加载预训练权重覆盖）
model = Gemma3Model(GEMMA3_CONFIG_270M)
# 打印模型结构，可查看各层名称、子模块及参数形状
model

In [ ]:
# 用一条简单输入 [1, 2, 3] 快速跑一次前向传播，验证模型结构和各层张量形状是否正确
# unsqueeze(0) 增加 batch 维度，输入形状变为 (1, 3)
model(torch.tensor([1, 2, 3]).unsqueeze(0))

In [ ]:
# 统计模型全部参数（含 tok_emb.weight 与 out_head.weight，即便二者可能共享同一份数据）的元素总数
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# Account for weight tying
# 因为 Gemma3-270m 对词嵌入与输出层做了权重共享（weight tying，
# tok_emb.weight 与 out_head.weight 是同一份参数），直接累加 parameters() 会重复计入这份权重，
# 这里减去一次 tok_emb.weight 的元素数，得到“去重后”的真实参数量
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

In [ ]:
# 粗略估算模型在给定数据类型下所需的显存/内存（参数 + 梯度 + 缓冲区），用于评估硬件需求；
# 注意这只是估算（比如推理时通常不需要保存梯度），实际占用还与优化器状态、激活值等因素有关
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

# 分别按 float32 和 bfloat16 两种精度估算显存占用，对比精度切换带来的显存节省
print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

In [ ]:
# 自动选择可用计算设备：优先 CUDA GPU，其次 Apple Silicon 的 MPS 后端，最后回退到 CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# 把模型（含所有参数和缓冲区，如 RoPE 的 cos/sin 表）搬到选定设备上
model.to(device);

4. Load pretrained weights

In [ ]:
# 把从 HuggingFace safetensors 文件加载出的原始权重字典（键名遵循 HF Gemma3 命名规范，
# 如 "model.layers.0.self_attn.q_proj.weight"）逐一拷贝赋值给本 notebook 自定义的 Gemma3Model 各子模块参数
def load_weights_into_gemma(model, param_config, params):

    # 内部工具函数：先校验形状是否匹配（避免因模型结构写错导致权重错位），
    # 再用 copy_ 原地拷贝数值到目标参数中（不改变参数对象本身）
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}")

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    # Embedding weights
    # 词嵌入权重
    if "model.embed_tokens.weight" in params:
        model.tok_emb.weight = assign(
            model.tok_emb.weight,
            params["model.embed_tokens.weight"],
            "model.embed_tokens.weight",
        )

    # Iterate over transformer layers
    # 逐层遍历，按层号从 params 中取出对应的 HF 键名并赋值
    for l in range(param_config["n_layers"]):
        block = model.blocks[l]
        att = block.att
        # Attention projections
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight",
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight",
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight",
        )
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight",
        )
        # QK normalization weights
        # QK-Norm 对应的 RMSNorm 权重（仅当 qk_norm=True 且权重文件中存在对应键时才会用到）
        att.q_norm.scale = assign(
            att.q_norm.scale,
            params[f"model.layers.{l}.self_attn.q_norm.weight"],
            f"model.layers.{l}.self_attn.q_norm.weight",
        )
        att.k_norm.scale = assign(
            att.k_norm.scale,
            params[f"model.layers.{l}.self_attn.k_norm.weight"],
            f"model.layers.{l}.self_attn.k_norm.weight",
        )
        # Feed forward weights
        # 注意 HF 命名与本实现命名的对应关系：
        # gate_proj -> fc1（门控分支），up_proj -> fc2（上投影分支），down_proj -> fc3（下投影分支）
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight",
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight",
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight",
        )
        # LayerNorm weights
        block.input_layernorm.scale = assign(
            block.input_layernorm.scale,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight",
        )
        block.post_attention_layernorm.scale = assign(
            block.post_attention_layernorm.scale,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight",
        )
        # Pre‑ and post‑feed forward norms
        pre_key = f"model.layers.{l}.pre_feedforward_layernorm.weight"
        post_key = f"model.layers.{l}.post_feedforward_layernorm.weight"
        if pre_key in params:
            block.pre_feedforward_layernorm.scale = assign(
                block.pre_feedforward_layernorm.scale,
                params[pre_key],
                pre_key,
            )
        if post_key in params:
            block.post_feedforward_layernorm.scale = assign(
                block.post_feedforward_layernorm.scale,
                params[post_key],
                post_key,
            )

    # Final LayerNorm
    if "model.norm.weight" in params:
        model.final_norm.scale = assign(
            model.final_norm.scale,
            params["model.norm.weight"],
            "model.norm.weight",
        )
    # Output head
    # 如果权重文件中单独提供了 lm_head.weight（输出层权重），直接加载
    if "lm_head.weight" in params:
        model.out_head.weight = assign(
            model.out_head.weight,
            params["lm_head.weight"],
            "lm_head.weight",
        )
        # 否则说明该模型采用了权重共享设计——输出层直接复用词嵌入矩阵，
        # 这是很多小型语言模型（包括 Gemma3-270m）常见的节省参数做法，
        # 与前面 cell 中“减去 tok_emb 权重数量”的处理相呼应
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")

In [ ]:
# 从 HuggingFace Hub 下载 Gemma3 预训练权重（safetensors 格式），
# 并调用 load_weights_into_gemma 把权重灌入前面自定义的模型
# Uncomment and run the following code if you are executing the notebook for the first time

#from huggingface_hub import login
#login()
# 用于读取分片权重的 index json 文件（仅当模型被切分成多个 safetensors 分片时才需要）
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download

# 选择模型规模，这里固定使用 270m —— Gemma3 系列中最小的版本
CHOOSE_MODEL = "270m"

# 根据前面设置的 USE_INSTRUCT_MODEL，决定下载指令微调版还是基础预训练版对应的 HF 仓库
if USE_INSTRUCT_MODEL:
    repo_id = f"google/gemma-3-{CHOOSE_MODEL}-it"
else:
    repo_id = f"google/gemma-3-{CHOOSE_MODEL}"


# 本地缓存目录，直接以仓库名命名
local_dir = Path(repo_id).parts[-1]

# 270m 模型体积小，权重通常只有单个 safetensors 文件，直接下载即可
if CHOOSE_MODEL == "270m":
    weights_file = hf_hub_download(
        repo_id=repo_id,
        filename="model.safetensors",
        local_dir=local_dir,
    )
    weights_dict = load_file(weights_file)
    # 若是更大的模型（权重被切分为多个分片文件），先下载整个仓库快照，
    # 再读取 index.json 中的 weight_map，找出每个参数分别存放在哪个分片文件里，逐个加载后合并
else:
    repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
    index_path = os.path.join(repo_dir, "model.safetensors.index.json")
    with open(index_path, "r") as f:
        index = json.load(f)

    weights_dict = {}
    for filename in set(index["weight_map"].values()):
        shard_path = os.path.join(repo_dir, filename)
        shard = load_file(shard_path)
        weights_dict.update(shard)

# 执行加载：把下载好的 HF 权重字典灌入模型
load_weights_into_gemma(model, GEMMA3_CONFIG_270M, weights_dict)
# 加载完成后再次确保模型在目标设备上
model.to(device)
# 释放原始权重字典占用的内存/显存
del weights_dict

3. Load tokenizer

In [ ]:
# 基于 HuggingFace tokenizers 库实现的 Gemma 分词器封装，
# 负责文本与 token id 的编解码，以及 Gemma 风格对话模板的拼接
from tokenizers import Tokenizer


class GemmaTokenizer:
    def __init__(self, tokenizer_file_path: str):
        # 加载 tokenizer.json 中保存的完整分词器配置（词表 + 分词规则等）
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))

        # Gemma 使用的一组特殊 token：<bos> 句首、<eos> 句尾、<pad> 填充、
        # <start_of_turn>/<end_of_turn> 用于分隔对话中的每一轮（角色 + 内容）
        self.bos_token = "<bos>"
        self.eos_token = "<eos>"
        self.pad_token = "<pad>"
        self.start_of_turn_token = "<start_of_turn>"
        self.end_of_turn_token = "<end_of_turn>"

        # 提前查出各特殊 token 对应的 id，方便后续在生成时判断是否遇到结束符
        self.bos_token_id = self._tok.token_to_id(self.bos_token)
        self.eos_token_id = self._tok.token_to_id(self.eos_token)
        self.pad_token_id = self._tok.token_to_id(self.pad_token)
        self.start_of_turn_token_id = self._tok.token_to_id(self.start_of_turn_token)
        self.end_of_turn_token_id = self._tok.token_to_id(self.end_of_turn_token)

        self.add_bos_token = True
        self.add_eos_token = False
        self.clean_up_tokenization_spaces = False

    # 编码：把文本转成 token id 列表
    def encode(self, text: str, add_special_tokens: bool = True) -> list[int]:
        return self._tok.encode(text, add_special_tokens=add_special_tokens).ids

    # 解码：把 token id（或单个 id）转换回文本
    def decode(self, ids: list[int] | int, skip_special_tokens: bool = False) -> str:
        if isinstance(ids, int):
            ids = [ids]
        return self._tok.decode(ids, skip_special_tokens=skip_special_tokens)

    # 按 Gemma 的对话模板格式拼接多轮对话：每一轮用
    # <start_of_turn>{role}\n{content}<end_of_turn>\n 包裹；
    # 注意 HF 中的 assistant 角色在 Gemma 模板里要写成 "model"
    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=False):
        text = ""
        for message in messages:
            role = message["role"]
            if role == "assistant":
                role = "model"
            content = message["content"]
            text += f"{self.start_of_turn_token}{role}\n{content}{self.end_of_turn_token}\n"

        # 若需要生成回复，末尾补上 "<start_of_turn>model\n"，提示模型以模型身份续写
        if add_generation_prompt:
            text += f"{self.start_of_turn_token}model\n"

        if tokenize:
            return self.encode(text)
        return text


# 顶层辅助函数：对单轮用户输入快速套用对话模板
# 注意：此函数与上面 GemmaTokenizer 类的同名方法是两个不同的对象，不会冲突，
# 但命名重复，调用时要留意区分 tokenizer.apply_chat_template（类方法）与这里的全局函数
def apply_chat_template(user_text):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False,
        add_generation_prompt=True,
    )
# 优先使用之前下载模型权重时创建的本地目录下的 tokenizer.json
tokenizer_file_path = os.path.join(local_dir, "tokenizer.json")
# 若本地没有，则从 HuggingFace Hub 单独下载 tokenizer.json
if not os.path.exists(tokenizer_file_path):
    try:
        tokenizer_file_path = hf_hub_download(repo_id=repo_id, filename="tokenizer.json", local_dir=local_dir)
    except Exception as e:
        print(f"Warning: failed to download tokenizer.json: {e}")
        tokenizer_file_path = "tokenizer.json"

# 实例化分词器
tokenizer = GemmaTokenizer(tokenizer_file_path=tokenizer_file_path)
# 待生成的用户问题
prompt = "Give me a short introduction to large language models."
# 套用对话模板，把裸文本包装成 Gemma 期望的对话格式字符串
prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
)


# 把套好模板的文本编码成 token id 列表，作为模型的输入
input_token_ids = tokenizer.encode(prompt)
# 反解码回文本，用于人工检查模板拼接和编解码是否正确
text = tokenizer.decode(input_token_ids)
text

5. Generate text

In [ ]:
# 定义一个最基础的、逐 token 流式生成（贪心解码）的函数，并用它对前面构造好的 prompt 生成回复
# Optionally use torch.compile for an extra speed-up
# model = torch.compile(model)
# 每次前向传播后只取最后一个位置的 logits，用 argmax 做贪心解码（不做采样/温度/top-k），
# 并通过 yield 把新生成的 token 实时“流式”返回给调用者
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):

    # 切换到 eval 模式（关闭 dropout 等训练专用行为）
    model.eval()
    # 生成阶段不需要梯度，关闭梯度记录以节省显存、加速推理
    with torch.no_grad():
        # 最多生成 max_new_tokens 个新 token
        for _ in range(max_new_tokens):
            # 对当前完整序列做一次前向传播（注意：这是最朴素的实现，每步都重新计算整个序列，
            # 没有使用 KV 缓存，速度较慢，仅作教学演示）；只取最后一个位置的 logits 用于预测下一个 token，
            # 形状 (b, vocab_size)
            out = model(token_ids)[:, -1]
            # 贪心解码：取概率最大（argmax）的 token 作为下一个预测，形状 (b, 1)
            next_token = torch.argmax(out, dim=-1, keepdim=True)

            # 若生成的 token 是结束符（这里用 Gemma 的 <end_of_turn>），提前停止生成
            if (eos_token_id is not None
                   and torch.all(next_token == eos_token_id)):
               break

            # 把新生成的 token 实时返回给调用方，实现流式输出
            yield next_token

            # 把新 token 拼接到已有序列末尾，作为下一步前向传播的输入（序列长度逐步增长）
            token_ids = torch.cat([token_ids, next_token], dim=1)
# 把之前编码好的 prompt token id 列表转换成张量并放到目标设备上，形状 (1, prompt_len)
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)


# 重置 GPU 显存峰值统计，方便之后测量本次生成实际耗费的显存
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


# 调用流式生成函数，逐 token 接收并即时打印结果
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    eos_token_id=tokenizer.end_of_turn_token_id
):
    # 去掉 batch 维度并转成 python 列表/整数，便于传给 tokenizer 解码
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

# 生成结束后，统计并打印本次推理过程中 GPU 显存的峰值占用
if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"\n\nGPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")